# Peptide Discovery v2 — Google Colab (T4 GPU)

**実行手順:**
1. ランタイム → ランタイムのタイプを変更 → **T4 GPU** を選択
2. 各セルを上から順に実行
3. Cell 6 で表示される URL をブラウザで開く

> Boltz モデルウェイト (~5 GB) を Google Drive にキャッシュすることで、2回目以降の起動を高速化できます。

In [ ]:
# ── Cell 1: GPU 確認 ──────────────────────────────────────────────────────────
import subprocess
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                        capture_output=True, text=True)
if result.returncode == 0:
    print('✅ GPU 検出:', result.stdout.strip())
else:
    print('❌ GPU が検出されません。ランタイムのタイプを T4 GPU に変更してください。')
    raise SystemExit('GPU required')

In [ ]:
# ── Cell 2: Google Drive マウント（モデルウェイトのキャッシュ用）────────────────
# キャッシュ不要な場合はこのセルをスキップしてください。
# キャッシュを使うと 2 回目以降の起動が ~5 分短縮されます。

USE_DRIVE_CACHE = True  # False にするとセッションごとに再ダウンロード

if USE_DRIVE_CACHE:
    from google.colab import drive
    drive.mount('/content/drive')
    BOLTZ_CACHE_DIR = '/content/drive/MyDrive/boltz_cache'
    import os
    os.makedirs(BOLTZ_CACHE_DIR, exist_ok=True)
    # boltz が ~/.boltz を参照するのでシンボリックリンクを貼る
    if not os.path.exists(os.path.expanduser('~/.boltz')):
        os.symlink(BOLTZ_CACHE_DIR, os.path.expanduser('~/.boltz'))
    print('✅ Drive キャッシュ設定完了:', BOLTZ_CACHE_DIR)
else:
    print('ℹ️  Drive キャッシュなし（セッションごとに再ダウンロード）')

In [ ]:
# ── Cell 3: リポジトリ取得 ────────────────────────────────────────────────────
# GitHub Personal Access Token を入力してください。
# 取得方法: GitHub → Settings → Developer settings → Personal access tokens → Tokens (classic)
# 必要なスコープ: repo (読み取り)

import getpass
import os

GITHUB_USER  = 'rkikuchi-bis'
GITHUB_REPO  = 'peptide_v2'
GITHUB_TOKEN = getpass.getpass('GitHub Personal Access Token: ')

repo_url = f'https://{GITHUB_TOKEN}@github.com/{GITHUB_USER}/{GITHUB_REPO}.git'

if os.path.exists(f'/content/{GITHUB_REPO}'):
    print('既存のディレクトリを更新します...')
    !cd /content/{GITHUB_REPO} && git pull
else:
    !git clone {repo_url} /content/{GITHUB_REPO}

os.chdir(f'/content/{GITHUB_REPO}')
print('✅ 作業ディレクトリ:', os.getcwd())

In [ ]:
# ── Cell 4: 依存関係インストール ──────────────────────────────────────────────
# Colab の既存 torch を使いつつ boltz などを追加インストールします。
# 所要時間: 約 3〜5 分

import subprocess, sys

packages = [
    'biopython==1.84',
    'streamlit>=1.36.0',
    'py3dmol>=2.0.0',
    'gemmi>=0.6',
    'altair>=5.0',
    'huggingface-hub>=0.20',
    'einops>=0.7',
    'biotite>=0.38',
    'boltz==2.2.1',
    'prodigy-prot',
    'pyngrok',
]

print('パッケージをインストール中...')
result = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '--quiet'] + packages,
    capture_output=True, text=True
)
if result.returncode != 0:
    print('STDERR:', result.stderr[-2000:])
    raise RuntimeError('インストール失敗')
print('✅ インストール完了')

In [ ]:
# ── Cell 5: Boltz モデルウェイト事前ダウンロード ──────────────────────────────
# 初回のみ実行（Drive キャッシュあれば 2 回目以降はスキップされます）。
# boltz1_conf.ckpt (~3.3 GB) + ccd.pkl (~200 MB)
# 所要時間: 約 5〜10 分（初回）

import os
from pathlib import Path

boltz_cache = Path(os.path.expanduser('~/.boltz'))
ckpt_path = boltz_cache / 'boltz1_conf.ckpt'

if ckpt_path.exists():
    size_gb = ckpt_path.stat().st_size / 1e9
    print(f'✅ キャッシュ済み: {ckpt_path} ({size_gb:.1f} GB) — スキップ')
else:
    print('モデルウェイトをダウンロード中（初回のみ、約 5〜10 分）...')
    # ダミー予測で boltz に自動ダウンロードさせる
    import tempfile, yaml
    dummy_yaml = {
        'version': 1,
        'sequences': [{
            'protein': {
                'id': 'A',
                'sequence': 'ACDEFGHIKLMNPQRSTVWY'
            }
        }]
    }
    with tempfile.TemporaryDirectory() as tmpdir:
        yaml_path = Path(tmpdir) / 'dummy.yaml'
        with open(yaml_path, 'w') as f:
            yaml.dump(dummy_yaml, f)
        result = subprocess.run(
            ['boltz', 'predict', str(yaml_path),
             '--out_dir', tmpdir,
             '--model', 'boltz1',
             '--accelerator', 'gpu',
             '--sampling_steps', '1',
             '--use_msa_server'],
            capture_output=True, text=True
        )
    if ckpt_path.exists():
        print('✅ ダウンロード完了')
    else:
        print('⚠️  ダウンロードに失敗した可能性があります。次のセルに進んでください（アプリ起動時に自動ダウンロードされます）。')

In [ ]:
# ── Cell 6: Streamlit 起動 + ngrok トンネル ───────────────────────────────────
# ngrok の無料アカウントが必要です: https://ngrok.com
# Authtoken: https://dashboard.ngrok.com/get-started/your-authtoken

import getpass, subprocess, time, threading
from pyngrok import ngrok, conf

NGROK_TOKEN = getpass.getpass('ngrok Authtoken: ')
conf.get_default().auth_token = NGROK_TOKEN

# Streamlit をバックグラウンド起動
streamlit_proc = subprocess.Popen(
    ['streamlit', 'run', 'app.py',
     '--server.port', '8501',
     '--server.headless', 'true',
     '--server.enableCORS', 'false',
     '--server.enableXsrfProtection', 'false'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT
)

# 起動待ち
print('Streamlit 起動中...')
time.sleep(5)

# ngrok トンネル開通
public_url = ngrok.connect(8501)
print('\n' + '='*60)
print('✅ アプリが起動しました！')
print(f'   URL: {public_url}')
print('='*60)
print('\n⚠️  このセルを停止するとアプリも終了します。')
print('   Colab セッションは最大 90 分で自動切断されます。')

# ログをリアルタイム表示
def stream_logs():
    for line in streamlit_proc.stdout:
        print('[streamlit]', line.decode().rstrip())

log_thread = threading.Thread(target=stream_logs, daemon=True)
log_thread.start()
streamlit_proc.wait()

## トラブルシューティング

| 症状 | 対処 |
|------|------|
| `GPU required` エラー | ランタイム → タイプを変更 → T4 GPU |
| ngrok `ERR_NGROK_108` | ngrok アカウントで既存トンネルを切断 |
| `boltz` コマンドが見つからない | Cell 4 を再実行後、ランタイムを再起動 |
| モデルウェイトが毎回ダウンロードされる | Cell 2 で `USE_DRIVE_CACHE = True` に設定 |
| Colab セッション切れ | 再度 Cell 3→6 を実行（Drive キャッシュあれば ~2 分で復旧） |